# SERDES 系统完整流程详解

本文通过可视化的方式，逐步演示 SERDES (Serializer/Deserializer) 系统的每一个关键环节。

## 什么是 SERDES？

SERDES 是**串行器/解串器**(Serializer/Deserializer) 的缩写，它是一种将并行数据转换为串行数据发送，再将接收的串行数据转换回并行数据的硬件电路。

## 本文演示流程

1. **数据生成** - 产生原始比特流
2. **串行化** - 并行转串行
3. **发送端预加重 (FFE)** - 补偿信道损耗
4. **信道传输** - 模拟物理信道特性
5. **接收端均衡** - 恢复信号质量
6. **解串行化** - 串行转并行
7. **数据恢复** - 采样与判决

---

## 第一步：导入库与基础设置

首先导入必要的 Python 库，并定义一些通用的绘图函数。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lfilter, butter, freqz
from scipy import signal

# 设置绘图风格
plt.style.use('ggplot')

# 设置中文字体（根据系统调整）
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("库导入成功！")

## 第二步：生成原始比特流

SERDES 系统处理的是二进制数据。我们首先生成一段随机的比特序列作为测试数据。

In [ ]:
def generate_random_bits(num_bits, seed=42):
    """
    生成随机比特流
    
    参数:
        num_bits: 比特数量
        seed: 随机种子，保证结果可复现
    
    返回:
        包含 0 和 1 的 numpy 数组
    """
    np.random.seed(seed)
    bits = np.random.randint(0, 2, num_bits)
    return bits

# 生成 20 个随机比特用于演示
num_bits = 20
original_bits = generate_random_bits(num_bits)

print(f"生成的原始比特流：{original_bits}")

In [ ]:
# 可视化原始比特流
fig, ax = plt.subplots(figsize=(12, 3))
ax.stem(range(num_bits), original_bits, basefmt='k-')
ax.set_xlabel('比特索引')
ax.set_ylabel('比特值')
ax.set_title('原始随机比特流')
ax.set_ylim(-0.1, 1.1)
ax.set_yticks([0, 1])
ax.grid(True, alpha=0.3)

# 标注每个比特的值
for i, val in enumerate(original_bits):
    ax.text(i, val + 0.05, str(int(val)), ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 第三步：串行化 (Serializer)

将离散的比特转换为连续的模拟信号波形。每个比特用多个采样点表示，以便后续处理。

In [ ]:
def serialize_bits(bits, samples_per_bit=40):
    """
    将比特序列转换为模拟信号
    
    参数:
        bits: 原始比特数组
        samples_per_bit: 每个比特的采样点数（过采样率）
    
    返回:
        连续的模拟信号数组
    """
    # 使用 np.repeat 实现串行化：每个比特重复 samples_per_bit 次
    serialized = np.repeat(bits, samples_per_bit).astype(float)
    return serialized

# 设置过采样率
samples_per_bit = 40

# 执行串行化
tx_signal = serialize_bits(original_bits, samples_per_bit)

print(f"原始比特数：{len(original_bits)}")
print(f"串行化后采样点数：{len(tx_signal)}")
print(f"过采样率：{samples_per_bit} samples/bit")

In [ ]:
# 可视化串行化后的信号
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

# 完整波形
axes[0].plot(tx_signal, 'b-', linewidth=1)
axes[0].set_xlabel('采样点索引')
axes[0].set_ylabel('幅度')
axes[0].set_title('串行化后信号 (完整波形)')
axes[0].set_ylim(-0.2, 1.2)
axes[0].grid(True, alpha=0.3)

# 前 5 个比特的细节
view_bits = 5
view_samples = view_bits * samples_per_bit
axes[1].plot(tx_signal[:view_samples], 'b-', linewidth=2)
axes[1].set_xlabel('采样点索引')
axes[1].set_ylabel('幅度')
axes[1].set_title(f'串行化后信号 (前{view_bits}个比特细节)')
axes[1].set_ylim(-0.2, 1.2)
axes[1].grid(True, alpha=0.3)

# 标注比特边界
for i in range(view_bits + 1):
    axes[1].axvline(x=i * samples_per_bit, color='r', linestyle='--', alpha=0.5, linewidth=1)

plt.tight_layout()
plt.show()

## 第四步：发送端预加重 (FFE - Feed Forward Equalizer)

由于高速信号在信道中传输时，高频分量衰减严重，导致信号失真。FFE 通过在发送端增强高频分量（信号跳变边缘）来补偿这种损耗。

### FFE 原理

FFE 是一个 FIR 滤波器，通常有 3 个抽头 (taps)：
- **Pre-cursor**: 主跳变前的预加重
- **Main**: 主抽头，承载主要能量
- **Post-cursor**: 主跳变后的去加重

公式：$y[n] = c_{-1} \cdot x[n+1] + c_0 \cdot x[n] + c_1 \cdot x[n-1]$

其中系数满足：$c_{-1} + c_0 + c_1 = 1$

In [ ]:
def apply_ffe(signal, pre_tap, post_tap):
    """
    应用 FFE 预加重
    
    参数:
        signal: 输入信号
        pre_tap: 预加重系数 (负值表示增强跳变)
        post_tap: 去加重系数 (负值表示抑制拖尾)
    
    返回:
        经过 FFE 处理后的信号
    """
    # 计算主抽头系数，保证总能量为 1
    main_tap = 1.0 - pre_tap - post_tap
    
    # FIR 滤波器系数
    fir_taps = [pre_tap, main_tap, post_tap]
    
    # 应用滤波
    filtered_signal = lfilter(fir_taps, [1.0], signal)
    
    return filtered_signal, fir_taps

# 设置 FFE 系数（典型值）
pre_cursor = -0.15  # 预加重
post_cursor = -0.10  # 去加重

# 应用 FFE
ffe_signal, fir_taps = apply_ffe(tx_signal, pre_cursor, post_cursor)

print(f"FFE 系数：Pre={pre_cursor}, Main={1-pre_cursor-post_cursor:.2f}, Post={post_cursor}")
print(f"系数和：{sum(fir_taps)} (应为 1)")

In [ ]:
# 可视化 FFE 效果
fig, axes = plt.subplots(3, 1, figsize=(14, 8))

view_samples = 8 * samples_per_bit

# 原始信号
axes[0].plot(tx_signal[:view_samples], 'b-', linewidth=2, label='原始信号')
axes[0].set_title('步骤 1: 串行化后的原始信号', fontsize=12)
axes[0].set_ylabel('幅度')
axes[0].set_ylim(-0.5, 1.5)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# FFE 处理后的信号
axes[1].plot(ffe_signal[:view_samples], 'r-', linewidth=2, label='FFE 输出')
axes[1].set_title(f'步骤 2: FFE 预加重后 (Pre={pre_cursor}, Post={post_cursor})', fontsize=12)
axes[1].set_ylabel('幅度')
axes[1].set_ylim(-0.5, 1.5)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 差异对比
diff = ffe_signal - tx_signal
axes[2].plot(diff[:view_samples], 'g-', linewidth=1.5, label='FFE 增量')
axes[2].fill_between(range(len(diff[:view_samples])), diff[:view_samples], alpha=0.3)
axes[2].set_title('步骤 3: FFE 产生的增量（在跳变边缘处最明显）', fontsize=12)
axes[2].set_xlabel('采样点索引')
axes[2].set_ylabel('幅度差')
axes[2].set_ylim(-0.5, 0.5)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 放大观察单个跳变
fig, ax = plt.subplots(figsize=(14, 4))
find_bit = 3  # 观察第 3 个比特附近的跳变
start = (find_bit - 1) * samples_per_bit
end = (find_bit + 2) * samples_per_bit

ax.plot(tx_signal[start:end], 'b-', linewidth=2, label='原始', alpha=0.7)
ax.plot(ffe_signal[start:end], 'r-', linewidth=2, label='FFE 后')
ax.set_xlabel('采样点索引（相对）')
ax.set_ylabel('幅度')
ax.set_title(f'单个跳变边缘的 FFE 效果放大图 (比特{find_bit-1} -> 比特{find_bit})')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 第五步：信道传输

信号通过物理信道（PCB 走线、电缆等）时，会受到以下影响：

1. **插入损耗 (Insertion Loss)** - 高频分量衰减更大
2. **反射 (Reflection)** - 阻抗不匹配导致
3. **串扰 (Crosstalk)** - 相邻信号干扰
4. **色散 (Dispersion)** - 不同频率传播速度不同

这里我们用一阶低通滤波器来简化模拟信道的低通特性。

In [ ]:
def model_channel(signal, cutoff_normalized):
    """
    模拟信道传输（低通滤波器）
    
    参数:
        signal: 输入信号
        cutoff_normalized: 归一化截止频率 (0~1，越小表示信道带宽越窄)
    
    返回:
        经过信道后的信号
    """
    # 设计一阶 Butterworth 低通滤波器
    b, a = butter(1, cutoff_normalized)
    
    # 应用滤波
    filtered = lfilter(b, a, signal)
    
    return filtered, (b, a)

# 设置信道截止频率（归一化）
# 值越小，信道带宽越窄，损耗越严重
channel_cutoff = 0.05

# 应用信道模型
channel_response, channel_coeffs = model_channel(ffe_signal, channel_cutoff)

print(f"信道截止频率（归一化）: {channel_cutoff}")
print(f"滤波器系数 b: {channel_coeffs[0]}")
print(f"滤波器系数 a: {channel_coeffs[1]}")

In [ ]:
# 可视化信道效果
fig, axes = plt.subplots(3, 1, figsize=(14, 8))

view_samples = 8 * samples_per_bit

# FFE 输出
axes[0].plot(ffe_signal[:view_samples], 'r-', linewidth=2, label='FFE 输出')
axes[0].set_title('步骤 1: FFE 预加重后的信号', fontsize=12)
axes[0].set_ylabel('幅度')
axes[0].set_ylim(-0.5, 1.5)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 信道输出
axes[1].plot(channel_response[:view_samples], 'g-', linewidth=2, label='信道输出')
axes[1].set_title(f'步骤 2: 经过信道后 (cutoff={channel_cutoff})', fontsize=12)
axes[1].set_ylabel('幅度')
axes[1].set_ylim(-0.5, 1.5)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 对比
axes[2].plot(ffe_signal[:view_samples], 'r--', linewidth=1, label='FFE 输出', alpha=0.5)
axes[2].plot(channel_response[:view_samples], 'g-', linewidth=2, label='信道输出')
axes[2].set_title('步骤 3: 信道对信号的影响（波形变圆滑，边缘变缓）', fontsize=12)
axes[2].set_xlabel('采样点索引')
axes[2].set_ylabel('幅度')
axes[2].set_ylim(-0.5, 1.5)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 第六步：接收端均衡 (CTLE)

CTLE (Continuous Time Linear Equalizer) 是接收端常用的均衡技术，它通过提升高频分量来补偿信道的低频损耗。

这里我们用高通滤波器特性来模拟 CTLE 的效果。

In [ ]:
def apply_ctle(signal, high_freq_gain=2.0):
    """
    应用 CTLE 均衡
    
    简化实现：使用高通滤波器提升高频分量
    
    参数:
        signal: 输入信号
        high_freq_gain: 高频增益
    
    返回:
        均衡后的信号
    """
    # 使用一阶高通滤波器
    b_hp, a_hp = butter(1, 0.1, btype='high')
    high_freq = lfilter(b_hp, a_hp, signal)
    
    # 原始信号 + 高频增强
    equalized = signal + high_freq_gain * high_freq
    
    # 归一化到合理范围
    equalized = equalized / (1 + high_freq_gain)
    
    return equalized

# 应用 CTLE
ctle_signal = apply_ctle(channel_response, high_freq_gain=1.5)

print(f"CTLE 高频增益：1.5")

In [ ]:
# 可视化 CTLE 效果
fig, axes = plt.subplots(3, 1, figsize=(14, 8))

view_samples = 8 * samples_per_bit

# 信道输出
axes[0].plot(channel_response[:view_samples], 'g-', linewidth=2, label='信道输出')
axes[0].set_title('步骤 1: 经过信道后的信号', fontsize=12)
axes[0].set_ylabel('幅度')
axes[0].set_ylim(-0.5, 1.5)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# CTLE 输出
axes[1].plot(ctle_signal[:view_samples], 'purple', linewidth=2, label='CTLE 输出')
axes[1].set_title('步骤 2: CTLE 均衡后（边缘变陡）', fontsize=12)
axes[1].set_ylabel('幅度')
axes[1].set_ylim(-0.5, 1.5)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 对比
axes[2].plot(channel_response[:view_samples], 'g--', linewidth=1, label='信道输出', alpha=0.5)
axes[2].plot(ctle_signal[:view_samples], 'purple', linewidth=2, label='CTLE 输出')
axes[2].set_title('步骤 3: CTLE 恢复了信号的边缘陡度', fontsize=12)
axes[2].set_xlabel('采样点索引')
axes[2].set_ylabel('幅度')
axes[2].set_ylim(-0.5, 1.5)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 第七步：采样与判决

接收端需要在正确的时刻对信号进行采样，并根据阈值判决是 0 还是 1。

关键问题：
1. **时钟恢复** - 确定采样时刻
2. **采样相位** - 在眼图张开最大的位置采样
3. **判决阈值** - 通常是信号幅度的中点

In [ ]:
def sample_and_decision(signal, samples_per_bit, threshold=0.5, phase_offset=0):
    """
    采样并判决
    
    参数:
        signal: 接收的信号
        samples_per_bit: 每个比特的采样点数
        threshold: 判决阈值
        phase_offset: 采样相位偏移 (0~1，0.5 表示在比特中心采样)
    
    返回:
        判决后的比特序列
    """
    num_bits = len(signal) // samples_per_bit
    recovered_bits = []
    
    # 在每个比特的中心位置采样
    sample_point = int(samples_per_bit * (0.5 + phase_offset))
    
    for i in range(num_bits):
        start = i * samples_per_bit
        sample_idx = start + sample_point
        
        if sample_idx < len(signal):
            sample_value = signal[sample_idx]
            # 判决：大于阈值为 1，否则为 0
            bit = 1 if sample_value > threshold else 0
            recovered_bits.append(bit)
    
    return np.array(recovered_bits), sample_point

# 执行采样判决
recovered_bits, sample_point = sample_and_decision(ctle_signal, samples_per_bit)

print(f"采样点位置：{sample_point} (每个比特{samples_per_bit}个采样点)")
print(f"原始比特：{original_bits[:len(recovered_bits)]}")
print(f"恢复比特：{recovered_bits}")

In [ ]:
# 可视化采样过程
fig, ax = plt.subplots(figsize=(14, 5))

view_bits = 10
view_samples = view_bits * samples_per_bit

# 绘制信号
ax.plot(ctle_signal[:view_samples], 'b-', linewidth=2, label='接收信号')

# 绘制判决阈值
ax.axhline(y=0.5, color='r', linestyle='--', label='判决阈值 (0.5)')

# 绘制采样点
sample_values = []
for i in range(view_bits):
    sample_idx = i * samples_per_bit + sample_point
    if sample_idx < len(ctle_signal[:view_samples]):
        sample_values.append(ctle_signal[sample_idx])
        ax.plot(sample_idx, ctle_signal[sample_idx], 'go', markersize=10, label='采样点' if i == 0 else '')
        # 画垂直线到阈值
        ax.plot([sample_idx, sample_idx], [0.5, ctle_signal[sample_idx]], 'g:', alpha=0.5)

ax.set_xlabel('采样点索引')
ax.set_ylabel('幅度')
ax.set_title(f'采样与判决过程（在比特中心{sample_point}处采样）')
ax.set_ylim(-0.2, 1.2)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 第八步：误码率计算

比较原始比特和恢复的比特，计算误码率 (BER - Bit Error Rate)。

In [ ]:
def calculate_ber(original, recovered):
    """
    计算误码率
    
    参数:
        original: 原始比特序列
        recovered: 恢复的比特序列
    
    返回:
        误码率
    """
    # 确保长度一致
    min_len = min(len(original), len(recovered))
    original = original[:min_len]
    recovered = recovered[:min_len]
    
    # 计算误码数
    errors = np.sum(original != recovered)
    ber = errors / min_len
    
    return ber, errors

# 计算误码率
ber, num_errors = calculate_ber(original_bits, recovered_bits)

print(f"="*50)
print(f"SERDES 系统传输结果")
print(f"="*50)
print(f"传输比特数：{len(original_bits)}")
print(f"误码数：{num_errors}")
print(f"误码率 (BER): {ber:.6f}")
if ber == 0:
    print("✓ 完美传输！")
else:
    print("✗ 存在误码")
print(f"="*50)

## 第九步：眼图分析

眼图是评估高速串行信号质量的最重要工具。通过叠加多个比特的波形，可以直观地看到：

- **眼高** - 垂直张开度，反映噪声容限
- **眼宽** - 水平张开度，反映时序容限
- **交叉点** - 上升/下降沿交汇处

眼睛张开越大，信号质量越好。

In [15]:
def plot_eye_diagram(signal, samples_per_bit, title, ax):
    """
    绘制眼图
    
    眼图通过叠加 2 个比特长度的波形来观察信号质量
    """
    num_samples = len(signal)
    num_bits = num_samples // samples_per_bit
    
    # 每次取 2 个比特的跨度
    step = samples_per_bit
    segment_length = 2 * samples_per_bit
    
    for i in range(0, num_samples - segment_length, step):
        segment = signal[i : i + segment_length]
        ax.plot(np.arange(len(segment)), segment, color='teal', alpha=0.1)
    
    ax.set_title(title, fontsize=12)
    ax.set_ylim(-0.5, 1.5)
    ax.set_xlim(0, segment_length)
    ax.grid(True, alpha=0.3)
    
    # 标注关键位置
    ax.axhline(y=0.5, color='r', linestyle='--', alpha=0.5)
    ax.axvline(x=samples_per_bit, color='r', linestyle='--', alpha=0.5)

In [ ]:
# 绘制各阶段的眼图对比
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. 理想信号眼图
plot_eye_diagram(tx_signal, samples_per_bit, "眼图 1: 发送端理想信号", axes[0, 0])

# 2. FFE 后眼图
plot_eye_diagram(ffe_signal, samples_per_bit, "眼图 2: FFE 预加重后", axes[0, 1])

# 3. 信道后眼图
plot_eye_diagram(channel_response, samples_per_bit, "眼图 3: 经过信道后（眼睛闭合）", axes[1, 0])

# 4. CTLE 后眼图
plot_eye_diagram(ctle_signal, samples_per_bit, "眼图 4: CTLE 均衡后（眼睛张开）", axes[1, 1])

plt.tight_layout()
plt.show()

## 第十步：交互式参数调整

使用交互控件实时调整参数，观察 SERDES 系统的响应。

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

@interact(
    pre_tap=widgets.FloatSlider(min=-0.3, max=0.0, step=0.02, value=-0.15, description='Pre-cursor:'),
    post_tap=widgets.FloatSlider(min=-0.3, max=0.0, step=0.02, value=-0.1, description='Post-cursor:'),
    channel_bw=widgets.FloatSlider(min=0.01, max=0.2, step=0.01, value=0.05, description='信道带宽:'),
    ctle_gain=widgets.FloatSlider(min=0.5, max=3.0, step=0.2, value=1.5, description='CTLE 增益:')
)
def run_interactive_sim(pre_tap, post_tap, channel_bw, ctle_gain):
    """
    交互式 SERDES 仿真
    """
    # 1. 生成数据
    bits = generate_random_bits(100)
    tx = serialize_bits(bits, samples_per_bit)
    
    # 2. FFE
    ffe_out, _ = apply_ffe(tx, pre_tap, post_tap)
    
    # 3. 信道
    ch_out, _ = model_channel(ffe_out, channel_bw)
    
    # 4. CTLE
    ctle_out = apply_ctle(ch_out, high_freq_gain=ctle_gain)
    
    # 5. 采样判决
    rec_bits, _ = sample_and_decision(ctle_out, samples_per_bit)
    ber, errs = calculate_ber(bits, rec_bits)
    
    # 绘图
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 时域波形
    view_lim = 10 * samples_per_bit
    axes[0, 0].plot(tx[:view_lim], 'k--', alpha=0.5, label='理想')
    axes[0, 0].plot(ch_out[:view_lim], 'b-', label='接收', linewidth=2)
    axes[0, 0].set_title('时域波形')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 眼图 - 发送端
    plot_eye_diagram(ffe_out, samples_per_bit, f'FFE 输出眼图', axes[0, 1])
    
    # 眼图 - 接收端
    plot_eye_diagram(ctle_out, samples_per_bit, f'CTLE 输出眼图', axes[1, 0])
    
    # 直方图
    ax_hist = axes[1, 1]
    ax_hist.hist(ctle_out[:2000], bins=100, color='blue', alpha=0.7)
    ax_hist.axvline(x=0.5, color='r', linestyle='--', label='阈值')
    ax_hist.set_title('信号幅度分布')
    ax_hist.legend()
    ax_hist.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"参数：Pre={pre_tap}, Post={post_tap}, BW={channel_bw}, CTLE={ctle_gain}")
    print(f"误码数：{errs}, 误码率：{ber:.6f}")

## 总结

通过本文的逐步演示，我们完整展示了 SERDES 系统的每个关键环节：

| 步骤 | 模块 | 功能 | 关键参数 |
|------|------|------|----------|
| 1 | 数据生成 | 产生随机比特流 | 比特数 |
| 2 | 串行化 | 并行转串行 | 过采样率 |
| 3 | FFE | 发送端预加重 | Pre/Post cursor |
| 4 | 信道 | 模拟传输损耗 | 截止频率 |
| 5 | CTLE | 接收端均衡 | 高频增益 |
| 6 | 采样判决 | 数据恢复 | 采样相位 |

### 关键要点

1. **FFE 和 CTLE 配合** - 发送端预加重和接收端均衡需要协同工作
2. **眼图是核心指标** - 眼图张开度直接决定系统能否正常工作
3. **信道带宽限制** - 物理信道的低通特性是高速传输的主要瓶颈
4. **采样相位很重要** - 在眼图中心采样能获得最大噪声容限

### 扩展阅读

- [低通滤波器对方波的影响](./低通滤波器对方波的影响.ipynb)
- [低通滤波器为何导致符号展宽](./低通滤波器为何导致符号展宽.ipynb)

---

![Colab](https://colab.research.google.com/assets/colab-badge.svg) [在 Colab 中打开](https://colab.research.google.com/github/yudonghai789/presonal_Blog/blob/master/高速传输 (serdes)/serdes 系统详解.ipynb)